# Khối D: Đánh giá ngưỡng và AUROC (14 bệnh)

Từ `logits_val.npy` / `labels_val.npy`: sigmoid, ngưỡng tối ưu F1, AUROC + CI bootstrap, F1 macro/micro, xuất Excel. Chạy lần lượt 3 ô (khoảng 1,5 phút; ô cuối là bước bootstrap).

## 1. Nối Drive và cấu hình đường dẫn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ---- Sửa đường dẫn nếu khác ----
# Thư mục chứa logits_val.npy, labels_val.npy (và logits_test.npy, labels_test.npy nếu có)
DATA_DIR = '/content/drive/MyDrive/CXR_Project/runs/densenet121_baseline/'
# File Excel đầu ra (lưu thẳng vào Drive)
OUT = '/content/drive/MyDrive/CXR_Project/results/threshold_eval.xlsx'


## 2. Hàm tính toán (Mục 8-12) và hàm xuất Excel

In [ ]:
import os

import numpy as np
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from sklearn.metrics import f1_score, precision_recall_curve, roc_auc_score

DISEASES = [
    'Atelectasis', 'Cardiomegaly', 'Effusion', 'Infiltration', 'Mass', 'Nodule',
    'Pneumonia', 'Pneumothorax', 'Consolidation', 'Edema', 'Emphysema',
    'Fibrosis', 'Pleural_Thickening', 'Hernia',
]
N_BOOT, SEED, FEW_POS = 1000, 0, 50   # dưới FEW_POS ca dương: cảnh báo "ít ca"



# ---------- Mục 1: sigmoid (ổn định số học) ----------
def sigmoid(z):
    z = np.asarray(z, dtype=np.float64)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    e = np.exp(z[~pos])
    out[~pos] = e / (1.0 + e)
    return out


# ---------- Mục 2: ngưỡng tối ưu F1 bằng precision_recall_curve ----------
def best_f1_threshold(y, p):
    prec, rec, thr = precision_recall_curve(y, p)
    prec, rec = prec[:-1], rec[:-1]            # khớp độ dài với thr
    denom = prec + rec
    f1 = np.divide(2 * prec * rec, denom, out=np.zeros_like(denom), where=denom > 0)
    k = int(np.argmax(f1))
    return float(thr[k]), float(f1[k])         # dự đoán dương khi p >= thr


def f1_at(y, p, t):
    return f1_score(y, (p >= t).astype(int), zero_division=0)


# ---------- Mục 3: F1 macro / micro ----------
def macro_micro(Y, P, thr):
    thr = np.broadcast_to(np.asarray(thr, dtype=float), (Y.shape[1],))
    pred = (P >= thr[None, :]).astype(int)
    Yi = Y.astype(int)
    return (f1_score(Yi, pred, average='macro', zero_division=0),
            f1_score(Yi, pred, average='micro', zero_division=0))


def auroc_each(Y, P):
    return np.array([roc_auc_score(Y[:, c], P[:, c]) for c in range(Y.shape[1])])


# ---------- Mục 4: bootstrap AUROC (lấy mẫu lại ảnh, dùng chung cho 14 lớp) ----------
def bootstrap_auroc(Y, P, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    n, C = Y.shape
    stats = np.full((n_boot, C), np.nan)
    for b in range(n_boot):
        i = rng.integers(0, n, n)
        yb, pb = Y[i], P[i]
        for c in range(C):
            s = yb[:, c].sum()
            if 0 < s < n:                      # bỏ lớp không có đủ cả hai loại
                stats[b, c] = roc_auc_score(yb[:, c], pb[:, c])
    ci = np.nanpercentile(stats, [2.5, 97.5], axis=0)
    mean_stats = np.nanmean(stats, axis=1)     # AUROC trung bình của mỗi lần lấy mẫu
    ci_mean = np.percentile(mean_stats, [2.5, 97.5])
    return ci, ci_mean


def main():
    Zv = np.load(os.path.join(DATA_DIR, 'logits_val.npy'))
    Yv = np.load(os.path.join(DATA_DIR, 'labels_val.npy')).astype(int)
    assert Zv.shape == Yv.shape and Zv.shape[1] == 14
    Pv = sigmoid(Zv)
    n_val = len(Yv)

    has_test = os.path.exists(os.path.join(DATA_DIR, 'logits_test.npy'))
    if has_test:
        Pt = sigmoid(np.load(os.path.join(DATA_DIR, 'logits_test.npy')))
        Yt = np.load(os.path.join(DATA_DIR, 'labels_test.npy')).astype(int)

    # --- tính toán ---
    auc = auroc_each(Yv, Pv)
    ci, ci_mean = bootstrap_auroc(Yv, Pv)
    best = [best_f1_threshold(Yv[:, c], Pv[:, c]) for c in range(14)]
    thr_opt = np.array([b[0] for b in best])
    f1_05 = np.array([f1_at(Yv[:, c], Pv[:, c], 0.5) for c in range(14)])
    f1_opt = np.array([b[1] for b in best])
    macro05, micro05 = macro_micro(Yv, Pv, 0.5)
    macro_o, micro_o = macro_micro(Yv, Pv, thr_opt)
    prev = Yv.mean(0)
    npos = Yv.sum(0)
    f1_allpos = 2 * prev / (1 + prev)          # F1 của mô hình luôn nói "có bệnh" (Mục 8)

    # kiểm tra chéo
    assert np.allclose(f1_opt, [f1_at(Yv[:, c], Pv[:, c], thr_opt[c]) for c in range(14)])
    assert np.isclose(f1_opt.mean(), macro_o) and np.isclose(f1_05.mean(), macro05)

    if has_test:
        prev_t = Yt.mean(0)
        auc_t = auroc_each(Yt, Pt)
        f1_t05 = np.array([f1_at(Yt[:, c], Pt[:, c], 0.5) for c in range(14)])
        f1_topt = np.array([f1_at(Yt[:, c], Pt[:, c], thr_opt[c]) for c in range(14)])
        f1_tbest = np.array([best_f1_threshold(Yt[:, c], Pt[:, c])[1] for c in range(14)])
        macro_t05, micro_t05 = macro_micro(Yt, Pt, 0.5)
        macro_to, micro_to = macro_micro(Yt, Pt, thr_opt)

    order = np.argsort(-prev)                  # bệnh phổ biến -> hiếm

    # --- in bảng ra màn hình ---
    print(f'Val: {n_val} ảnh | bootstrap B={N_BOOT}, seed={SEED}')
    print(f'{"Bệnh":<19}{"Tỉ lệ":>7}{"Dương":>6}{"AUROC":>8}{"CI 95%":>17}{"Ngưỡng":>8}{"F1@0.5":>8}{"F1@opt":>8}')
    for c in order:
        print(f'{DISEASES[c]:<19}{prev[c]*100:6.2f}%{npos[c]:6d}{auc[c]:8.4f}'
              f'  [{ci[0,c]:.3f}, {ci[1,c]:.3f}]{thr_opt[c]:8.3f}{f1_05[c]:8.4f}{f1_opt[c]:8.4f}')
    print(f'AUROC TB = {auc.mean():.4f}  CI [{ci_mean[0]:.4f}, {ci_mean[1]:.4f}]')
    print(f'Macro F1: 0.5 -> {macro05:.4f} | ngưỡng tối ưu -> {macro_o:.4f}')
    print(f'Micro F1: 0.5 -> {micro05:.4f} | ngưỡng tối ưu -> {micro_o:.4f}')

    # ================= Excel =================
    wb = Workbook()
    HEAD = PatternFill('solid', fgColor='1F3864')
    WARN = PatternFill('solid', fgColor='FCE4D6')
    GREY = PatternFill('solid', fgColor='F2F2F2')
    thin = Side(style='thin', color='BFBFBF')
    BOX = Border(left=thin, right=thin, top=thin, bottom=thin)

    def style_header(ws, row, ncol):
        for j in range(1, ncol + 1):
            c = ws.cell(row, j)
            c.font = Font(bold=True, color='FFFFFF')
            c.fill = HEAD
            c.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            c.border = BOX

    def widths(ws, ws_widths):
        for j, w in enumerate(ws_widths, 1):
            ws.column_dimensions[get_column_letter(j)].width = w

    # ---- Sheet 1: kết quả chính ----
    ws = wb.active
    ws.title = 'Kết quả (val)'
    ws['A1'] = 'Đánh giá ngưỡng và AUROC theo từng bệnh – tập validation (sắp theo tỉ lệ bệnh, phổ biến → hiếm)'
    ws['A1'].font = Font(bold=True, size=13)
    ws['A2'] = (f'N = {n_val} ảnh. Ngưỡng tối ưu chọn trên chính tập val nên "F1 ngưỡng tối ưu" là con số lạc quan '
                f'(đặc biệt với bệnh hiếm). CI 95% = bootstrap {N_BOOT} lần, phân vị 2,5% và 97,5%.')
    ws['A2'].alignment = Alignment(wrap_text=True, vertical='top')
    ws.merge_cells('A2:K2')
    ws.row_dimensions[2].height = 32
    cols = ['Tên bệnh', 'Tỉ lệ bệnh (val)', 'Số ca dương', 'AUROC', 'Khoảng tin cậy 95% AUROC',
            'Độ rộng CI', 'Ngưỡng tối ưu', 'F1 (ngưỡng 0,5)', 'F1 (ngưỡng tối ưu)',
            'F1 nếu luôn dự đoán "có bệnh"', 'Ghi chú']
    HR = 4
    for j, h in enumerate(cols, 1):
        ws.cell(HR, j, h)
    style_header(ws, HR, len(cols))
    ws.row_dimensions[HR].height = 45
    r = HR + 1
    for c in order:
        note = ''
        if npos[c] < FEW_POS:
            note = (f'CHỈ {npos[c]} ca dương trong val: CI rất rộng, ngưỡng tối ưu và F1 không đáng tin '
                    f'(F1 có thể bị thổi phồng).')
        if DISEASES[c] != 'Hernia' and 0 < npos[c] < 150:
            note = note or 'Ít ca dương (<150): CI tương đối rộng.'
        row = [DISEASES[c], prev[c], int(npos[c]), auc[c], f'[{ci[0,c]:.3f} ; {ci[1,c]:.3f}]',
               ci[1, c] - ci[0, c], thr_opt[c], f1_05[c], f1_opt[c], f1_allpos[c], note]
        for j, v in enumerate(row, 1):
            cell = ws.cell(r, j, float(v) if isinstance(v, (np.floating,)) else v)
            cell.border = BOX
            cell.alignment = Alignment(vertical='center', wrap_text=(j == 11),
                                       horizontal='left' if j in (1, 11) else 'center')
        ws.cell(r, 2).number_format = '0.00%'
        for j in (4, 6, 7, 8, 9, 10):
            ws.cell(r, j).number_format = '0.000'
        if npos[c] < FEW_POS:
            for j in range(1, len(cols) + 1):
                ws.cell(r, j).fill = WARN
            ws.row_dimensions[r].height = 48
        r += 1
    # dòng trung bình
    ws.cell(r, 1, 'Trung bình (macro)')
    ws.cell(r, 4, float(auc.mean()))
    ws.cell(r, 5, f'[{ci_mean[0]:.3f} ; {ci_mean[1]:.3f}]')
    ws.cell(r, 6, float(ci_mean[1] - ci_mean[0]))
    ws.cell(r, 8, float(macro05))
    ws.cell(r, 9, float(macro_o))
    for j in range(1, len(cols) + 1):
        c_ = ws.cell(r, j)
        c_.font = Font(bold=True)
        c_.fill = GREY
        c_.border = BOX
        c_.alignment = Alignment(horizontal='left' if j == 1 else 'center')
        if j in (4, 6, 8, 9):
            c_.number_format = '0.000'
    r += 2
    ws.cell(r, 1, 'Tô cam = bệnh dưới 50 ca dương. Hernia chỉ 18 ca dương nên mọi số liệu của bệnh này '
                  'chỉ mang tính tham khảo. Cột "F1 nếu luôn dự đoán có bệnh" = 2π/(1+π), mức nền để so với F1 của mô hình.')
    ws.merge_cells(start_row=r, start_column=1, end_row=r, end_column=11)
    ws.cell(r, 1).alignment = Alignment(wrap_text=True, vertical='top')
    ws.row_dimensions[r].height = 32
    widths(ws, [20, 12, 10, 9, 24, 10, 11, 12, 13, 17, 55])
    ws.freeze_panes = ws.cell(HR + 1, 2)

    # ---- Sheet 2: so sánh ngưỡng 0,5 và ngưỡng tối ưu ----
    w2 = wb.create_sheet('So sánh tổng hợp (val)')
    hdr = ['Chỉ số', 'Ngưỡng cố định 0,5', 'Ngưỡng tối ưu từng bệnh', 'Chênh lệch']
    for j, h in enumerate(hdr, 1):
        w2.cell(1, j, h)
    style_header(w2, 1, 4)
    rows = [('F1 macro', macro05, macro_o), ('F1 micro', micro05, micro_o)]
    for i, (n_, a, b) in enumerate(rows, 2):
        for j, v in enumerate([n_, a, b, b - a], 1):
            c_ = w2.cell(i, j, v)
            c_.border = BOX
            if j > 1:
                c_.number_format = '0.0000'
                c_.alignment = Alignment(horizontal='center')
    w2.cell(5, 1, f'AUROC trung bình = {auc.mean():.4f}, CI 95% bootstrap [{ci_mean[0]:.4f} ; {ci_mean[1]:.4f}] '
                  f'(AUROC không phụ thuộc ngưỡng).')
    w2.cell(6, 1, 'Lưu ý: F1 ngưỡng tối ưu được chọn và đo trên cùng tập val nên chênh lệch này là mức tăng tối đa, '
                  'không phải mức tăng sẽ thấy trên dữ liệu mới.')
    w2.cell(8, 1, 'F1 theo từng bệnh (cùng thứ tự với sheet 1)').font = Font(bold=True)
    for j, h in enumerate(['Tên bệnh', 'F1 @0,5', 'F1 @ngưỡng tối ưu', 'Tăng'], 1):
        w2.cell(9, j, h)
    style_header(w2, 9, 4)
    for i, c in enumerate(order, 10):
        for j, v in enumerate([DISEASES[c], f1_05[c], f1_opt[c], f1_opt[c] - f1_05[c]], 1):
            c_ = w2.cell(i, j, float(v) if j > 1 else v)
            c_.border = BOX
            if j > 1:
                c_.number_format = '0.0000'
                c_.alignment = Alignment(horizontal='center')
    widths(w2, [30, 20, 24, 14])

    # ---- Sheet 3: test (tham khảo) ----
    if has_test:
        w3 = wb.create_sheet('Test (tham khảo)')
        w3['A1'] = ('Áp dụng ngưỡng đã chọn trên VAL vào TEST (không chọn lại ngưỡng trên test). '
                    'Tỉ lệ bệnh test khác val nên F1 hai tập không so trực tiếp được.')
        w3['A1'].font = Font(bold=True)
        w3.merge_cells('A1:J1')
        w3['A1'].alignment = Alignment(wrap_text=True, vertical='top')
        w3.row_dimensions[1].height = 32
        th = ['Tên bệnh', 'Tỉ lệ val', 'Tỉ lệ test', 'Tỉ lệ test / val', 'AUROC test',
              'Ngưỡng (từ val)', 'F1 test @0,5', 'F1 test @ngưỡng val', 'F1 test tốt nhất có thể',
              'F1 test nếu luôn dự đoán "có bệnh"']
        for j, h in enumerate(th, 1):
            w3.cell(3, j, h)
        style_header(w3, 3, len(th))
        w3.row_dimensions[3].height = 45
        for i, c in enumerate(order, 4):
            vals = [DISEASES[c], prev[c], prev_t[c], prev_t[c] / prev[c], auc_t[c], thr_opt[c],
                    f1_t05[c], f1_topt[c], f1_tbest[c], 2 * prev_t[c] / (1 + prev_t[c])]
            for j, v in enumerate(vals, 1):
                c_ = w3.cell(i, j, float(v) if j > 1 else v)
                c_.border = BOX
                if j > 1:
                    c_.alignment = Alignment(horizontal='center')
                    c_.number_format = '0.00%' if j in (2, 3) else '0.000'
            if DISEASES[c] == 'Hernia' or abs(prev_t[c] / prev[c] - 1) > 0.5:
                for j in range(1, len(th) + 1):
                    w3.cell(i, j).fill = WARN
        k = 4 + 14
        w3.cell(k, 1, 'Macro'); w3.cell(k, 5, float(auc_t.mean()))
        w3.cell(k, 7, float(macro_t05)); w3.cell(k, 8, float(macro_to))
        w3.cell(k + 1, 1, 'Micro'); w3.cell(k + 1, 7, float(micro_t05)); w3.cell(k + 1, 8, float(micro_to))
        for rr in (k, k + 1):
            for j in range(1, len(th) + 1):
                c_ = w3.cell(rr, j)
                c_.font = Font(bold=True); c_.fill = GREY; c_.border = BOX
                if j > 1:
                    c_.number_format = '0.000'; c_.alignment = Alignment(horizontal='center')
        w3.cell(k + 3, 1, 'Tô cam = bệnh có tỉ lệ test lệch hơn 50% so với val (hoặc Hernia, rất hiếm). '
                          'Khi tỉ lệ đổi, precision và F1 ở cùng một ngưỡng đổi theo dù mô hình không đổi.')
        w3.merge_cells(start_row=k + 3, start_column=1, end_row=k + 3, end_column=10)
        w3.cell(k + 3, 1).alignment = Alignment(wrap_text=True, vertical='top')
        w3.row_dimensions[k + 3].height = 32
        widths(w3, [20, 10, 10, 12, 11, 13, 13, 15, 16, 20])
        w3.freeze_panes = 'B4'

    # ---- Sheet 4: ghi chú phương pháp ----
    w4 = wb.create_sheet('Ghi chú')
    notes = [
        'Phương pháp (theo Mục 8-12 của tài liệu công thức)',
        '1. Xác suất = sigmoid(logit).',
        '2. Ngưỡng tối ưu: precision_recall_curve của sklearn, tính F1 = 2PR/(P+R) tại mọi ngưỡng, lấy ngưỡng có F1 lớn nhất. Dự đoán dương khi p ≥ ngưỡng.',
        '3. F1 macro = trung bình F1 14 bệnh; F1 micro = F1 từ TP/FP/FN gộp chung. AUROC bằng roc_auc_score.',
        f'4. CI 95% AUROC: lấy mẫu lại ảnh có hoàn lại {N_BOOT} lần (seed {SEED}), cùng một bộ chỉ số ảnh dùng cho cả 14 bệnh, '
        'lấy phân vị 2,5% và 97,5%. CI của AUROC trung bình lấy từ chính AUROC trung bình của mỗi lần lấy mẫu.',
        '',
        'Cảnh báo',
        f'- Hernia chỉ có {npos[DISEASES.index("Hernia")]} ca dương trong val: CI AUROC rất rộng '
        f'({ci[0,13]:.2f} đến {ci[1,13]:.2f}). Theo mô phỏng ở Mục 12, CI bootstrap với ~18 ca dương chỉ phủ giá trị thật khoảng 90% '
        'thay vì 95%, nên thực tế còn không chắc chắn hơn con số này.',
        '- Ngưỡng tối ưu của bệnh hiếm chọn trên rất ít ca dương nên F1 trên val thổi phồng (Mục 9 mô phỏng: val 0,123 so với test 0,056 cho bệnh kiểu Hernia).',
        '- F1 ngưỡng tối ưu trong sheet 1 và 2 được chọn và đo trên cùng val, nên lạc quan. Sheet "Test (tham khảo)" cho số trung thực hơn.',
        f'- Tỉ lệ bệnh khác giữa val và test: Pneumothorax {prev[7]*100:.1f}% (val) so với {prev_t[7]*100:.1f}% (test)' if has_test else '',
        '  (ngược với ghi chú ban đầu "3% so với 10%": số liệu thật cho thấy val là 3%, test là 10%). AUROC không đổi theo tỉ lệ bệnh nhưng precision và F1 thì có.',
    ]
    for i, t in enumerate(notes, 1):
        c_ = w4.cell(i, 1, t)
        c_.alignment = Alignment(wrap_text=True, vertical='top')
        if t in ('Phương pháp (theo Mục 8-12 của tài liệu công thức)', 'Cảnh báo'):
            c_.font = Font(bold=True, size=12)
    w4.column_dimensions['A'].width = 130

    os.makedirs(os.path.dirname(OUT) or '.', exist_ok=True)
    wb.save(OUT)
    print('Đã lưu', OUT)


## 3. Chạy và xuất Excel

In [ ]:
main()

from google.colab import files
files.download(OUT)  # tải thêm về máy (bỏ dòng này nếu không cần)
